# Validation du modèle — Identification de fragments parallèles

Ce notebook applique les modèles de régression logistique entraînés précédemment à des données comparables non alignées, préalablement converties en CSV par le script `prepare_validation_corpus.py`.

**Prérequis :** avoir exécuté `prepare_validation_corpus.py` qui produit dans `validation/csv/` :
```
validation/csv/
├── fr_gsw_alsacien.csv
├── fr_gsw_mulhousien.csv
├── fr_gsw_strasbourgeois.csv
├── fr_co.csv
└── fr_oc.csv
```

Chaque CSV contient les colonnes : `variete`, `langue`, `paire`, `fr_sentence`, `lr_sentence`.

**Pipeline :**
1. Chargement des modèles et des CSV
2. Calcul des variables d'apprentissage
3. Inférence (modèle binaire + multiclasse)
4. Analyse des résultats par variété et par langue
5. Export CSV des paires candidates

## 0. Imports et configuration

In [ ]:
import os
import re
import unicodedata
import string
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

try:
    import jellyfish
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jellyfish", "-q"])
    import jellyfish

try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim
    _sbert_available = True
except ImportError:
    _sbert_available = False
    print("[AVERTISSEMENT] sentence-transformers non disponible — score SBERT mis à 0.")

# -------------------------------------------------------
# Configuration
# -------------------------------------------------------
CSV_DIR          = "validation/csv/"      # dossier contenant les CSV préparés
MODEL_BIN_PATH   = "logreg_binary.joblib"
MODEL_MC_PATH    = "logreg_multiclass.joblib"
SCALER_PATH      = "scaler.joblib"
SBERT_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
OUTPUT_CSV       = "validation_candidates.csv"

# Seuil de confiance minimum pour retenir une paire candidate
CONFIDENCE_THRESHOLD = 0.65

# CSV à charger — None = tous les CSV présents dans CSV_DIR
# Pour n'en charger que certains : ["fr_co.csv", "fr_oc.csv"]
CSV_FILES = None

FEATURE_COLS = [
    "sbert_score",
    "length_score",
    "jaccard_bigrams",
    "jaccard_trigrams",
    "jaccard_quadrigrams",
    "phonetic_jaccard",
]

print("Configuration chargée.")

## 1. Chargement des modèles

In [ ]:
scaler    = joblib.load(SCALER_PATH)
model_bin = joblib.load(MODEL_BIN_PATH)
model_mc  = joblib.load(MODEL_MC_PATH)
mc_classes = model_mc.classes_

print(f"  ✔ Scaler           : {SCALER_PATH}")
print(f"  ✔ Modèle binaire   : {MODEL_BIN_PATH}")
print(f"  ✔ Modèle multiclasse : {MODEL_MC_PATH}")
print(f"  Classes : {mc_classes}")

if _sbert_available:
    sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
    print(f"  ✔ SBERT            : {SBERT_MODEL_NAME}")

## 2. Chargement des CSV de validation

In [ ]:
def load_validation_csvs(csv_dir: str, files: list = None) -> pd.DataFrame:
    """
    Charge tous les CSV présents dans csv_dir (ou seulement ceux listés dans files).
    Retourne un DataFrame consolidé.
    Vérifie la présence des colonnes attendues et signale les fichiers manquants.
    """
    expected_cols = {"variete", "langue", "paire", "fr_sentence", "lr_sentence"}

    if not os.path.isdir(csv_dir):
        raise FileNotFoundError(
            f"Dossier CSV introuvable : {csv_dir}\n"
            f"Exécutez d'abord prepare_validation_corpus.py"
        )

    available = sorted(f for f in os.listdir(csv_dir) if f.endswith('.csv'))
    to_load   = files if files else available

    if not to_load:
        raise FileNotFoundError(f"Aucun CSV trouvé dans {csv_dir}")

    dfs = []
    for fname in to_load:
        fpath = os.path.join(csv_dir, fname)
        if not os.path.exists(fpath):
            print(f"  [MANQUANT] {fpath} — ignoré")
            continue
        df = pd.read_csv(fpath, encoding="utf-8-sig")
        missing = expected_cols - set(df.columns)
        if missing:
            print(f"  [ERREUR] {fname} — colonnes manquantes : {missing}")
            continue
        # Supprime les lignes avec des phrases vides
        before = len(df)
        df = df.dropna(subset=["fr_sentence", "lr_sentence"])
        df = df[df["fr_sentence"].str.strip().ne("") & df["lr_sentence"].str.strip().ne("")]
        dropped = before - len(df)
        print(f"  ✔ {fname:<35} {len(df):>7} paires"
              + (f"  ({dropped} lignes vides supprimées)" if dropped else ""))
        dfs.append(df)

    if not dfs:
        raise ValueError("Aucun CSV valide chargé.")

    return pd.concat(dfs, ignore_index=True)


print("Chargement des CSV de validation...\n")
df_validation = load_validation_csvs(CSV_DIR, CSV_FILES)

print(f"\nDataFrame consolidé : {len(df_validation)} paires")
print(f"Variétés présentes  : {df_validation['variete'].unique().tolist()}")
print(f"Langues présentes   : {df_validation['langue'].unique().tolist()}")

In [ ]:
# Statistiques descriptives par variété
stats = df_validation.groupby("variete").agg(
    langue=("langue", "first"),
    paires_totales=("fr_sentence", "count"),
    phrases_fr_uniques=("fr_sentence", "nunique"),
    phrases_lr_uniques=("lr_sentence", "nunique"),
    documents_uniques=("paire", "nunique"),
).reset_index()
print("=== Statistiques par variété ===")
print(stats.to_string(index=False))

## 3. Calcul des variables d'apprentissage

In [ ]:
def normalize(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.translate(str.maketrans("", "", string.punctuation)).strip()

def length_score(s1: str, s2: str) -> float:
    n1, n2 = len(s1.split()), len(s2.split())
    m = max(n1, n2)
    return 0.0 if m == 0 else abs(n1 - n2) / m

def char_ngrams(text: str, n: int) -> set:
    t = normalize(text)
    return set(t[i:i+n] for i in range(len(t) - n + 1))

def jaccard_index(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def jaccard_ngrams(s1: str, s2: str, n: int) -> float:
    return jaccard_index(char_ngrams(s1, n), char_ngrams(s2, n))

def metaphone_codes(text: str) -> set:
    codes = set()
    for word in normalize(text).split():
        try:
            code = jellyfish.metaphone(word)
            if code:
                codes.add(code)
        except Exception:
            pass
    return codes

def phonetic_jaccard(s1: str, s2: str) -> float:
    return jaccard_index(metaphone_codes(s1), metaphone_codes(s2))


def compute_features_batch(fr_sentences: list, lr_sentences: list,
                            sbert_model=None, batch_size: int = 64) -> np.ndarray:
    """
    Calcule le vecteur de variables pour une liste de paires.
    Retourne un array de forme (N, 6).
    """
    n = len(fr_sentences)
    sbert_scores = np.zeros(n)

    if sbert_model is not None:
        for i in tqdm(range(0, n, batch_size), desc="SBERT", leave=False):
            b_fr = fr_sentences[i:i+batch_size]
            b_lr = lr_sentences[i:i+batch_size]
            e_fr = sbert_model.encode(b_fr, convert_to_numpy=True, show_progress_bar=False)
            e_lr = sbert_model.encode(b_lr, convert_to_numpy=True, show_progress_bar=False)
            for j, (v1, v2) in enumerate(zip(e_fr, e_lr)):
                sbert_scores[i+j] = float(max(0.0, cos_sim([v1], [v2])[0][0]))

    rows = []
    for i, (fr, lr) in enumerate(zip(fr_sentences, lr_sentences)):
        rows.append([
            sbert_scores[i],
            length_score(fr, lr),
            jaccard_ngrams(fr, lr, 2),
            jaccard_ngrams(fr, lr, 3),
            jaccard_ngrams(fr, lr, 4),
            phonetic_jaccard(fr, lr),
        ])
    return np.array(rows)


print("Fonctions de calcul des variables définies.")

In [ ]:
print(f"Calcul des variables sur {len(df_validation)} paires...")

fr_list = df_validation["fr_sentence"].tolist()
lr_list = df_validation["lr_sentence"].tolist()

sbert_arg = sbert_model if _sbert_available else None
features  = compute_features_batch(fr_list, lr_list, sbert_model=sbert_arg)

# Ajout des variables au DataFrame
for i, col in enumerate(FEATURE_COLS):
    df_validation[col] = features[:, i]

print("Variables calculées et ajoutées au DataFrame.")
df_validation[FEATURE_COLS].describe().round(4)

## 4. Inférence

In [ ]:
features_sc  = scaler.transform(features)

# Modèle binaire
pred_bin     = model_bin.predict(features_sc)
proba_bin    = model_bin.predict_proba(features_sc)[:, 1]

# Modèle multiclasse
pred_mc_idx  = model_mc.predict(features_sc)
proba_mc_all = model_mc.predict_proba(features_sc)
conf_mc      = proba_mc_all[np.arange(len(pred_mc_idx)), pred_mc_idx]
label_mc     = [mc_classes[p] for p in pred_mc_idx]

df_validation["pred_binaire"]   = pred_bin
df_validation["confiance_bin"]  = proba_bin.round(4)
df_validation["pred_classe"]    = label_mc
df_validation["confiance_mc"]   = conf_mc.round(4)

print("Inférence terminée.")
print(f"\nDistribution des prédictions binaires :")
print(df_validation["pred_binaire"].value_counts().rename({0: "non-parallèle", 1: "parallèle"}).to_string())
print(f"\nDistribution des classes multiclasse :")
print(df_validation["pred_classe"].value_counts().to_string())

## 5. Analyse des résultats

### 5.1 Paires candidates

In [ ]:
df_candidates = df_validation[
    (df_validation["pred_binaire"] == 1) &
    (df_validation["confiance_bin"] >= CONFIDENCE_THRESHOLD)
].copy().sort_values("confiance_bin", ascending=False)

print(f"Paires candidates retenues (seuil ≥ {CONFIDENCE_THRESHOLD}) :")
print(f"  {len(df_candidates)} / {len(df_validation)} "
      f"({100*len(df_candidates)/max(len(df_validation),1):.1f}%)")

print("\nTop 10 paires candidates :")
for _, row in df_candidates.head(10).iterrows():
    print(f"\n  [{row['variete']}] {row['pred_classe']} | "
          f"conf={row['confiance_bin']:.2%} | SBERT={row['sbert_score']:.3f}")
    print(f"  FR : {row['fr_sentence'][:100]}")
    print(f"  LR : {row['lr_sentence'][:100]}")

### 5.2 Performances par variété et par langue

In [ ]:
def stats_for_group(group_col: str) -> pd.DataFrame:
    """Calcule les statistiques de validation pour un groupement donné."""
    rows = []
    for val in df_validation[group_col].unique():
        sub  = df_validation[df_validation[group_col] == val]
        cand = sub[
            (sub["pred_binaire"] == 1) &
            (sub["confiance_bin"] >= CONFIDENCE_THRESHOLD)
        ]
        par  = cand[cand["pred_classe"] == "parallel"]
        semi = cand[cand["pred_classe"] == "semi-parallel"]
        rows.append({
            group_col:               val,
            "Comparaisons":          len(sub),
            "Candidates":            len(cand),
            "Parallèles":            len(par),
            "Semi-parallèles":       len(semi),
            "Taux extraction (%)": round(100*len(cand)/max(len(sub),1), 2),
            "Conf. moy.": round(cand["confiance_bin"].mean(), 4) if len(cand) else 0,
            "SBERT moy.": round(cand["sbert_score"].mean(), 4)   if len(cand) else 0,
        })
    return pd.DataFrame(rows)


print("=== Par variété ===")
stats_variete = stats_for_group("variete")
print(stats_variete.to_string(index=False))

print("\n=== Par langue ===")
stats_langue = stats_for_group("langue")
print(stats_langue.to_string(index=False))

### 5.3 Visualisations

In [ ]:
# Distribution SBERT : toutes comparaisons vs candidates, par variété
varietes = df_validation["variete"].unique()
n = len(varietes)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=False)
if n == 1:
    axes = [axes]

for ax, var in zip(axes, varietes):
    sub  = df_validation[df_validation["variete"] == var]["sbert_score"]
    cand = df_candidates[df_candidates["variete"] == var]["sbert_score"]
    ax.hist(sub,  bins=30, alpha=0.45, color="steelblue", label="toutes")
    ax.hist(cand, bins=30, alpha=0.75, color="#2ecc71",   label="candidates")
    if len(cand):
        ax.axvline(cand.mean(), color="darkgreen", linestyle="--",
                   linewidth=1.5, label=f"moy={cand.mean():.2f}")
    ax.set_title(var.replace("fr_", ""), fontsize=9)
    ax.set_xlabel("Score SBERT")
    ax.set_ylabel("Fréquence")
    ax.legend(fontsize=7)

plt.suptitle("Distribution SBERT par variété — toutes comparaisons vs candidates",
             fontsize=10)
plt.tight_layout()
plt.savefig("validation_sbert_par_variete.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : validation_sbert_par_variete.png")

In [ ]:
# Barplot : candidates par variété et par classe
if len(df_candidates):
    pivot = (df_candidates
             .groupby(["variete", "pred_classe"])
             .size()
             .unstack(fill_value=0))
    col_order  = [c for c in ["parallel", "semi-parallel"] if c in pivot.columns]
    colors_cls = {"parallel": "#2ecc71", "semi-parallel": "#f39c12"}
    pivot[col_order].plot(
        kind="bar",
        color=[colors_cls[c] for c in col_order],
        figsize=(9, 4),
        edgecolor="white",
    )
    plt.title("Paires candidates par variété et par classe")
    plt.xlabel("Variété")
    plt.ylabel("Nombre de paires")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Classe")
    plt.tight_layout()
    plt.savefig("validation_candidates_par_variete.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : validation_candidates_par_variete.png")

In [ ]:
# Heatmap : SBERT moyen par variété × classe (candidates uniquement)
if len(df_candidates):
    pivot_heat = (df_candidates
                  .groupby(["variete", "pred_classe"])["sbert_score"]
                  .mean()
                  .unstack(fill_value=0))
    fig, ax = plt.subplots(figsize=(6, max(3, len(pivot_heat)*0.6)))
    sns.heatmap(pivot_heat, annot=True, fmt=".2f", cmap="YlGn",
                ax=ax, linewidths=0.5, vmin=0, vmax=1)
    ax.set_title("Score SBERT moyen par variété et classe (candidates)")
    ax.set_xlabel("Classe")
    ax.set_ylabel("Variété")
    plt.tight_layout()
    plt.savefig("validation_sbert_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardée : validation_sbert_heatmap.png")

In [ ]:
# Effet du seuil de confiance sur le nombre de candidates
thresholds   = np.arange(0.50, 0.96, 0.05)
n_cands_thr  = []
mean_sbert_thr = []

for thr in thresholds:
    c = df_validation[
        (df_validation["pred_binaire"] == 1) &
        (df_validation["confiance_bin"] >= thr)
    ]
    n_cands_thr.append(len(c))
    mean_sbert_thr.append(c["sbert_score"].mean() if len(c) else 0)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(thresholds, n_cands_thr, width=0.04, alpha=0.6,
        color="steelblue", label="Nb candidates")
ax2.plot(thresholds, mean_sbert_thr, 'o-', color="darkorange",
         linewidth=2, label="SBERT moyen")
ax1.axvline(CONFIDENCE_THRESHOLD, color="red", linestyle="--",
            linewidth=1.5, label=f"Seuil actuel = {CONFIDENCE_THRESHOLD}")
ax1.set_xlabel("Seuil de confiance")
ax1.set_ylabel("Nombre de paires candidates", color="steelblue")
ax2.set_ylabel("Score SBERT moyen", color="darkorange")
ax1.set_title("Effet du seuil de confiance")
lines = ax1.get_legend_handles_labels()
lines2 = ax2.get_legend_handles_labels()
ax1.legend(lines[0]+lines2[0], lines[1]+lines2[1], fontsize=8)
plt.tight_layout()
plt.savefig("validation_seuil_confiance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : validation_seuil_confiance.png")

## 6. Export CSV

In [ ]:
# Export global
df_candidates.sort_values(
    ["variete", "confiance_bin"], ascending=[True, False]
).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"  ✔ {OUTPUT_CSV} — {len(df_candidates)} paires")

# Export par variété
for var in df_candidates["variete"].unique():
    sub = df_candidates[df_candidates["variete"] == var]
    out = f"validation_candidates_{var}.csv"
    sub.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"  ✔ {out} — {len(sub)} paires")

## 7. Résumé final

In [ ]:
print("=" * 60)
print("RÉSUMÉ DE LA VALIDATION")
print("=" * 60)
print(f"Seuil de confiance        : {CONFIDENCE_THRESHOLD}")
print(f"Comparaisons totales      : {len(df_validation)}")
print(f"Paires candidates retenues: {len(df_candidates)} "
      f"({100*len(df_candidates)/max(len(df_validation),1):.1f}%)")
print()
print("--- Par variété ---")
print(stats_variete.to_string(index=False))
print()
print("--- Par langue ---")
print(stats_langue.to_string(index=False))